# 05 Test New Model Config

Эксперимент: обучаем event GRU/LSTM на компактном наборе event-признаков и сравниваем с rule baseline.

In [1]:
from pathlib import Path
import random
import sys

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.connector.data_fetcher import load_all_price_data
from src.features.event_detector import detect_events
from src.features.feature_pipeline import generate_features
from src.models.sequence_models import load_model_with_config
from src.models.training import train_direction_model
from src.strategy.backtest import build_trades, calculate_trade_metrics
from src.strategy.signal_generator import add_labels_for_metrics, generate_rule_based_signal_history, generate_signal_history

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_columns", None)

## Идея

Rule baseline уже силен, значит надо не добавлять шум, а дать модели маленький набор признаков вокруг события.

In [2]:
RUN_TRAINING = False
MODEL_TYPES = ["gru", "lstm"]

SELECTION_METRIC = "balanced_accuracy"
EPOCHS = 40
HORIZON = config.DEFAULT_HORIZON_CANDLES
LABEL_THRESHOLD = config.DEFAULT_LABEL_THRESHOLD
Q_CANDLES = 2000
CONFIDENCE_GRID = [0.50, 0.52, 0.55, 0.57, 0.60]

SIMPLE_FEATURE_COLUMNS = [
    "event_cusum_direction",
    "near_support",
    "near_resistance",
    "breakout_up",
    "breakout_down",
    "strong_range",
    "strong_body",
    "range_ratio_20",
    "body_ratio_20",
    "dist_to_prev_sup",
    "dist_to_prev_res",
    "dist_to_prev_sup_96",
    "dist_to_prev_res_96",
    "rsi_14_norm",
    "volatility_20",
    "time_sin",
    "time_cos",
]

SIMPLE_PARAMS = {
    "gru": {
        "learning_rate": 0.0015,
        "hidden_size": 64,
        "dropout": 0.25,
        "num_layers": 2,
        "batch_size": 128,
    },
    "lstm": {
        "learning_rate": 0.0015,
        "hidden_size": 64,
        "dropout": 0.30,
        "num_layers": 2,
        "batch_size": 128,
    },
}

In [3]:
price_df, loaded_files = load_all_price_data(PROJECT_ROOT / "data")
prepared_df = detect_events(generate_features(price_df))
print(f"loaded files={len(loaded_files)}, rows={len(price_df):,}, events={int(prepared_df['event'].sum()):,}")

loaded files=45, rows=277,105, events=21,684


In [4]:
def simple_paths(model_type: str):
    model_path = config.MODELS_DIR / f"simple_event_{model_type}_best.pth"
    scaler_path = config.MODELS_DIR / f"simple_event_{model_type}_scaler.pkl"
    config_path = config.MODELS_DIR / f"simple_event_{model_type}_config.pkl"
    return model_path, scaler_path, config_path


def train_simple_model(model_type: str):
    params = SIMPLE_PARAMS[model_type]
    model_path, scaler_path, config_path = simple_paths(model_type)
    result = train_direction_model(
        price_df=price_df,
        model_type=model_type,
        event_only=True,
        label_threshold=LABEL_THRESHOLD,
        horizon=HORIZON,
        epochs=EPOCHS,
        selection_metric=SELECTION_METRIC,
        model_path=model_path,
        scaler_path=scaler_path,
        feature_columns=SIMPLE_FEATURE_COLUMNS,
        **params,
    )
    joblib.dump(
        {
            "model_type": model_type,
            "input_size": len(SIMPLE_FEATURE_COLUMNS),
            "hidden_size": params["hidden_size"],
            "dropout": params["dropout"],
            "num_layers": params["num_layers"],
            "selection_metric": SELECTION_METRIC,
            "label_threshold": LABEL_THRESHOLD,
            "horizon": HORIZON,
            "feature_columns": SIMPLE_FEATURE_COLUMNS,
        },
        config_path,
    )
    return {
        "model": model_type,
        "best_valid_score": result["best_valid_score"],
        "test_accuracy": result["test_metrics"]["accuracy"],
        "test_balanced_accuracy": result["test_metrics"]["balanced_accuracy"],
        "test_f1": result["test_metrics"]["f1"],
        "model_path": str(model_path),
        "config_path": str(config_path),
    }

In [5]:
if RUN_TRAINING:
    train_rows = [train_simple_model(model_type) for model_type in MODEL_TYPES]
    display(pd.DataFrame(train_rows))
else:
    print("Обучение пропущено, используем сохраненные simple-модели.")

Обучение пропущено, используем сохраненные simple-модели.


In [6]:
def apply_confidence_threshold(signals: pd.DataFrame, threshold: float, require_event: bool = True, cusum_mode: str = "none") -> pd.DataFrame:
    if signals.empty:
        return signals.copy()
    result = signals.copy()
    event_allowed = result["event"].eq(1) if require_event else pd.Series(True, index=result.index)
    confident = result["confidence"].ge(threshold)

    if "event_cusum_direction" in result.columns:
        agrees_up = result["prediction"].eq("UP") & result["event_cusum_direction"].eq(1)
        agrees_down = result["prediction"].eq("DOWN") & result["event_cusum_direction"].eq(-1)
        agreement = agrees_up | agrees_down
    else:
        agreement = pd.Series(True, index=result.index)

    if cusum_mode == "agree":
        direction_filter = agreement
    elif cusum_mode == "disagree":
        direction_filter = ~agreement
    else:
        direction_filter = pd.Series(True, index=result.index)

    result["decision"] = "NO TRADE"
    result.loc[event_allowed & confident & direction_filter & result["prediction"].eq("UP"), "decision"] = "BUY"
    result.loc[event_allowed & confident & direction_filter & result["prediction"].eq("DOWN"), "decision"] = "SELL"
    return result


def model_metrics(signals: pd.DataFrame) -> dict:
    labeled = add_labels_for_metrics(signals, prepared_df, HORIZON, threshold=LABEL_THRESHOLD)
    labeled = labeled[labeled["event"].eq(1)].copy()
    if labeled.empty:
        return {"accuracy": 0.0, "balanced_accuracy": 0.0, "f1": 0.0, "precision": 0.0, "recall": 0.0, "classified": 0}
    y_true = labeled["actual"]
    y_pred = labeled["prediction"]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, pos_label="UP", zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label="UP", zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label="UP", zero_division=0),
        "classified": len(labeled),
    }


def summarize(name: str, signals: pd.DataFrame, confidence: float, cusum_mode: str = "none"):
    filtered = apply_confidence_threshold(
        signals,
        confidence,
        require_event=True,
        cusum_mode=cusum_mode,
    )
    trades = build_trades(filtered, prepared_df, horizon=HORIZON)
    trade_metrics = calculate_trade_metrics(trades)
    clf = model_metrics(filtered)
    return {
        "strategy": name,
        "confidence": confidence,
        "cusum_mode": cusum_mode,
        "accuracy": clf["accuracy"],
        "balanced_accuracy": clf["balanced_accuracy"],
        "f1": clf["f1"],
        "classified": clf["classified"],
        "trades": trade_metrics["Trades"],
        "total_return": trade_metrics["Total Return"],
        "winrate": trade_metrics["Win Rate"],
        "profit_factor": trade_metrics["Profit Factor"],
    }

In [7]:
rows = []

for model_type in MODEL_TYPES:
    model_path, scaler_path, config_path = simple_paths(model_type)
    model_config = joblib.load(config_path)
    model = load_model_with_config(model_path, config_path)
    scaler = joblib.load(scaler_path)
    raw_signals = generate_signal_history(
        price_df,
        threshold=0.50,
        max_rows=Q_CANDLES,
        model_type=model_type,
        require_event=True,
        model=model,
        scaler=scaler,
        feature_columns=model_config["feature_columns"],
    )
    for confidence in CONFIDENCE_GRID:
        rows.append(summarize(f"Simple Event + {model_type.upper()}", raw_signals, confidence))
        rows.append(summarize(f"Simple Event + {model_type.upper()} + CUSUM agree", raw_signals, confidence, cusum_mode="agree"))
        rows.append(summarize(f"Simple Event + {model_type.upper()} + CUSUM disagree", raw_signals, confidence, cusum_mode="disagree"))

rule_signals = generate_rule_based_signal_history(price_df, max_rows=Q_CANDLES)
rows.append(summarize("Rule baseline", rule_signals, 0.50))

comparison_df = pd.DataFrame(rows).sort_values(["total_return", "profit_factor"], ascending=[False, False])
display(comparison_df)

,strategy,confidence,cusum_mode,accuracy,balanced_accuracy,f1,classified,trades,total_return,winrate,profit_factor
30,Rule baseline,0.50,none,0.527778,0.516755,0.442623,144,163,0.01640,0.558282,1.259864
12,Simple Event + GRU,0.60,none,0.472222,0.483245,0.486486,144,9,0.00126,0.555556,1.445230
14,Simple Event + GRU + CUSUM disagree,0.60,disagree,0.472222,0.483245,0.486486,144,9,0.00126,0.555556,1.445230
27,Simple Event + LSTM,0.60,none,0.472222,0.483245,0.486486,144,4,0.00002,0.500000,1.010870
29,Simple Event + LSTM + CUSUM disagree,0.60,disagree,0.472222,0.483245,0.486486,144,4,0.00002,0.500000,1.010870
1,Simple Event + GRU + CUSUM agree,0.50,agree,0.472222,0.483245,0.486486,144,4,0.00000,0.500000,1.000000
4,Simple Event + GRU + CUSUM agree,0.52,agree,0.472222,0.483245,0.486486,144,0,0.00000,0.000000,0.000000
7,Simple Event + GRU + CUSUM agree,0.55,agree,0.472222,0.483245,0.486486,144,0,0.00000,0.000000,0.000000
10,Simple Event + GRU + CUSUM agree,0.57,agree,0.472222,0.483245,0.486486,144,0,0.00000,0.000000,0.000000
13,Simple Event + GRU + CUSUM agree,0.60,agree,0.472222,0.483245,0.486486,144,0,0.00000,0.000000,0.000000


## Что пробовать дальше

1. Подбирать `LABEL_THRESHOLD` и `HORIZON` вместе: текущий label может не совпадать с TP/SL.
2. Убрать LSTM, если GRU стабильно лучше: меньше параметров, меньше переобучения.
3. Добавить отдельный фильтр: торговать ML-сигнал только когда он совпадает с `event_cusum_direction`.
4. Проверять параметры на validation, а финальный результат смотреть только на test.